Выделяет на DEM непрерывную область океана

In [1]:
import arcpy
from arcpy.sa import Raster, RegionGroup, SetNull, MajorityFilter, ExtractByAttributes

arcpy.env.overwriteOutput = True

tandem = r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\GDBs\TanDEMX.gdb\TDX_EGM_Mosaic"
# Есть ещё один не связанный сектор океана, выделил его отдельно и объединил вручную
# tandem =  r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\GDBs\TanDEMX.gdb\tdx_egm_fill"
result_db = r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\GDBs\TanDEMX.gdb"
ocean_border = f"{result_db}\ocean_border"

# ПАРАМЕТРЫ
z_threshold = 0.1  # Высота в метрах, ниже которой считаем всё "потенциальной водой"
min_area_sq_km = 10.0      # Минимальная площадь куска океана (км²)

In [2]:
with arcpy.EnvManager(outputCoordinateSystem=arcpy.SpatialReference(32631), cellSize=tandem, scratchWorkspace=r"F:\temp"):

    print("1. Создание маски воды по высоте...")
    # Если высота < z_threshold, ставим 1, иначе NoData. 
    # Используем SetNull, чтобы суша стала прозрачной.
    water_mask = SetNull(Raster(tandem) > z_threshold, 1)

    print("2. Уборка шума (Majority Filter)...")
    # Убирает одиночные пиксели-выбросы (актуально для TanDEM-X)
    water_clean = MajorityFilter(water_mask, "EIGHT", "MAJORITY")

    print("3. Группировка регионов (Region Group)...")
    regions_raster = RegionGroup(water_clean, "EIGHT", "CROSS", "NO_LINK", 0)

    print("4. Фильтрация по площади...")
    # Получаем свойства растра напрямую из объекта
    cell_w = regions_raster.meanCellWidth
    cell_h = regions_raster.meanCellHeight
    pixel_area_m2 = cell_w * cell_h

    # Пересчитываем порог площади (км²) в количество пикселей (Count)
    target_area_m2 = min_area_sq_km * 1_000_000
    threshold_count = int(target_area_m2 / pixel_area_m2)

    print(f"   Площадь пикселя: {pixel_area_m2:.1f} м²")
    print(f"   Порог: {min_area_sq_km} км² = {threshold_count} пикселей")

    # ExtractByAttributes возвращает новый объект Raster, содержащий только те регионы,
    # где Count (количество пикселей) больше порога.
    ocean_regions = ExtractByAttributes(regions_raster, f"Count >= {threshold_count}")

    
    print("5. Конвертация чистого океана в полигоны...")
    # Сохраняем во временный вектор (в памяти или scratch), чтобы потом почистить
    temp_fc = r"memory\ocean_raw"
    
    arcpy.conversion.RasterToPolygon(
        in_raster=ocean_regions,
        out_polygon_features=temp_fc,
        simplify="SIMPLIFY",
        raster_field="Value",
        create_multipart_features="SINGLE_OUTER_PART"
    )
    print("6. Финальная постобработка геометрии...")
    # Заполняем пустоты
    arcpy.management.EliminatePolygonPart(
        in_features=temp_fc,
        out_feature_class=ocean_border,
        condition="AREA",
        part_area="2 Hectares",
        part_area_percent=0,
        part_option="CONTAINED_ONLY"
    )

    print("7. Очистка временных файлов...")
    # Удаляем временный вектор океана
    try:
        if arcpy.Exists(temp_fc):
            arcpy.management.Delete(temp_fc)
            print(f"   Временный файл удален: {temp_fc}")
    except Exception as e:
        print(f"   Внимание: не удалось удалить {temp_fc}. Ошибка: {e}")
        # Такое бывает, если слой все еще держит блокировку. 
        # Обычно при завершении скрипта блокировка спадает сама.
    
print("Готово. Океан выделен.")

1. Создание маски воды по высоте...
2. Уборка шума (Majority Filter)...
3. Группировка регионов (Region Group)...
4. Фильтрация по площади...
   Площадь пикселя: 935.1 м²
   Порог: 10.0 км² = 10694 пикселей
5. Конвертация чистого океана в полигоны...
6. Финальная постобработка геометрии...
7. Очистка временных файлов...
   Временный файл удален: memory\ocean_raw
Готово. Океан выделен.
